In [1]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# 1. 설정 및 초기화 (results_table.ipynb와 동일한 구조 + ADMET-Print, MMFRL 2개만 추가)
datasets = ['Kd', 'Ki']

base_path = '/data/rlawlsgurjh/work/gitlab/CUE/cue/results'
dta_base_path = '/data/rlawlsgurjh/work/gitlab/CUE/dta_results'

label_map = {
    'regression': ["PI", 'Ssel', "WS(alpha=2)", "RS(k=2)"],
}

# 분석 대상 지표
regression_candidates = ['rmse', 'r2', 'pcc']
# classification_candidates = ['auc_roc', 'auprc']

# 베이스라인 모델 설정
# format: 'dta_csv' (DTA 공통 규격), 'admet_csv' (MAE,RMSE,R2,PCC), 'mmfdl_summary' (metric,mean,std),
#         'new_csv' (ADMET-Print/MMFRL, {metric}_mean/{metric}_std 컬럼)
baseline_configs = {
    'DeepDTA':     {'path': os.path.join(dta_base_path, 'DeepDTA'), 'format': 'dta_csv'},
    'GraphDTA':    {'path': os.path.join(dta_base_path, 'GraphDTA'), 'format': 'dta_csv'},
    'TEFDTA':      {'path': os.path.join(dta_base_path, 'TEFDTA'), 'format': 'dta_csv'},
    'HMM-DTA':     {'path': os.path.join(dta_base_path, 'HMM-DTA'), 'format': 'dta_csv'},
    'ADMET-Print': {'path': '/data/rlawlsgurjh/work/gitlab/CUE/admet-print/results', 'format': 'new_csv'},
    'ADMET-AI':    {'path': '/data/rlawlsgurjh/work/gitlab/CUE/admet_ai/results/chemprop_rdkit', 'format': 'admet_csv'},
    'MMFDL':       {'path': '/data/rlawlsgurjh/work/gitlab/CUE/MMFDL/results/SGD/selectivity', 'format': 'mmfdl_summary'},
    'MMFRL':       {'path': '/data/rlawlsgurjh/work/gitlab/CUE/mmfrl/results/v1', 'format': 'new_csv'},
}

# 모델별 비교 대상 매핑 (Regression만 수행하는 모델 등 구분)
baseline_models = list(baseline_configs.keys())

In [2]:
def load_baseline_metric(model_name, dataset, target, config):
    """모델별 포맷에 맞춰 metrics 정보를 가져오고 이름 정규화 (평균/표준편차 포함).
    results_table.ipynb의 dta_csv/admet_csv/mmfdl_summary 로직에 ADMET-Print/MMFRL용
    'new_csv' 포맷만 추가한 버전이다."""
    fmt = config['format']
    base_dir = config['path']

    if model_name == 'MMFDL':
        metrics_path = os.path.join(base_dir, dataset, target, 'v1', 'summary_metric.csv')
    else:
        filename = 'metrics.csv' if fmt in ('dta_csv', 'new_csv') else 'metric.csv'
        metrics_path = os.path.join(base_dir, dataset, target, filename)

    if not os.path.exists(metrics_path):
        return None

    try:
        df = pd.read_csv(metrics_path)
        res = {}

        # 메트릭 이름 표준화 맵
        name_map = {
            'auc': 'auc_roc', 'auc-roc': 'auc_roc', 'auroc': 'auc_roc', 'au_roc': 'auc_roc',
            'au_prc': 'auprc', 'auprc': 'auprc'
        }

        if fmt == 'dta_csv':
            row = df[df.iloc[:, 0].str.contains(target.replace('_label', ''), na=False, regex=False)]
            if row.empty: row = df.iloc[:1]
            for col in row.columns:
                c_low = col.lower()
                is_std = c_low.endswith('_std')
                pure_col = c_low.replace('_std', '')
                norm_col = name_map.get(pure_col, pure_col)
                if norm_col in regression_candidates: # regression_candidates + classification_candidates:
                    key = f"{norm_col}_std" if is_std else norm_col
                    res[key] = row[col].values[0]

        elif fmt == 'admet_csv':
            m_row = df[df.iloc[:, 0] == 'mean']
            s_row = df[df.iloc[:, 0] == 'std']
            for col in df.columns:
                c_low = col.lower()
                norm_col = name_map.get(c_low, c_low)
                if norm_col in regression_candidates: # regression_candidates + classification_candidates:
                    if not m_row.empty: res[norm_col] = m_row[col].values[0]
                    if not s_row.empty: res[f"{norm_col}_std"] = s_row[col].values[0]

        elif fmt == 'mmfdl_summary':
            for _, r in df.iterrows():
                m_low = str(r['metric']).lower()
                norm_col = name_map.get(m_low, m_low)
                if norm_col in regression_candidates: # regression_candidates + classification_candidates:
                    res[norm_col] = r['mean']
                    res[f"{norm_col}_std"] = r['std']

        elif fmt == 'new_csv':
            row = df.iloc[0]
            for col in row.index:
                c_low = col.lower()
                if c_low.endswith('_mean'):
                    pure_col = c_low[:-5]
                    norm_col = name_map.get(pure_col, pure_col)
                    if norm_col in regression_candidates: # regression_candidates + classification_candidates:
                        res[norm_col] = row[col]
                elif c_low.endswith('_std'):
                    pure_col = c_low[:-4]
                    norm_col = name_map.get(pure_col, pure_col)
                    if norm_col in regression_candidates: # regression_candidates + classification_candidates:
                        res[f"{norm_col}_std"] = row[col]

        return res if res else None
    except:
        return None

raw_data_list = []
for d_name in datasets:
    for task_type, target_list in label_map.items():
        version = 'v1'
        for target in target_list:
            # 1. CUE(=cmfe) 결과 로드
            m_path = os.path.join(base_path, d_name, target, version, 'CMFE', 'metrics.csv')
            if os.path.exists(m_path):
                df_m = pd.read_csv(m_path)
                mlp_row = df_m[df_m['model_name'] == 'mlp']
                if not mlp_row.empty:
                    entry = {'Dataset_Group': d_name, 'Target': target, 'Modality': 'CUE', 'Task': task_type}
                    for col in mlp_row.columns:
                        if col in ['model_name', 'Unnamed: 0']: continue
                        norm_col = 'auc_roc' if col.lower() == 'auroc' else 'auc_roc_std' if col.lower() == 'auroc_std' else col.lower()
                        entry[norm_col] = mlp_row[col].values[0]
                    raw_data_list.append(entry)

            # 2. 베이스라인 모델 결과 로드 (DTA 4개 + ADMET-AI + MMFDL + ADMET-Print + MMFRL)
            for b_name, b_config in baseline_configs.items():
                b_metrics = load_baseline_metric(b_name, d_name, target, b_config)
                if b_metrics:
                    entry = {'Dataset_Group': d_name, 'Target': target, 'Modality': b_name, 'Task': task_type}
                    entry.update(b_metrics)
                    raw_data_list.append(entry)

df_all = pd.DataFrame(raw_data_list)

In [3]:
df_all.head(10)

,Dataset_Group,Target,Modality,Task,mse,mse_std,rmse,rmse_std,r2,r2_std,pcc,pcc_std,ci,ci_std
0,Kd,PI,CUE,regression,0.058444,0.004952,0.241524,0.010476,0.125334,0.055290,0.365724,0.092164,0.571147,0.050314
1,Kd,PI,DeepDTA,regression,NaN,NaN,0.348885,0.018091,-0.831704,0.222976,0.567114,0.031852,NaN,NaN
2,Kd,PI,GraphDTA,regression,NaN,NaN,0.337639,0.037180,-0.732145,0.410775,0.573663,0.035862,NaN,NaN
3,Kd,PI,TEFDTA,regression,NaN,NaN,0.349607,0.020546,-0.834385,0.173722,0.565755,0.037417,NaN,NaN
4,Kd,PI,HMM-DTA,regression,NaN,NaN,0.275571,0.042428,-0.152845,0.312975,0.543310,0.090161,NaN,NaN
5,Kd,PI,ADMET-Print,regression,NaN,NaN,0.269575,0.011580,-0.095575,0.148583,0.219042,0.047109,NaN,NaN
6,Kd,PI,ADMET-AI,regression,NaN,NaN,0.258313,0.010210,-0.005443,0.045507,0.181295,0.125199,NaN,NaN
7,Kd,PI,MMFDL,regression,NaN,NaN,0.263681,0.008250,-0.043000,0.059380,0.221761,0.052070,NaN,NaN
8,Kd,PI,MMFRL,regression,NaN,NaN,0.257565,0.009969,0.005171,0.058219,0.371817,0.057871,NaN,NaN
9,Kd,Ssel,CUE,regression,0.804734,0.124792,0.894523,0.067543,0.065169,0.106895,0.314014,0.104265,0.658580,0.016982


In [4]:
# 지표별 방향 표시 맵
metric_direction = {
    'rmse': '↓', 'r2': '↑', 'pcc': '↑'# , 'auc_roc': '↑', 'auprc': '↑'
}

# results_table.ipynb와 저장 경로가 겹치면 나중에 실행되는 쪽이 덮어써서 ADMET-Print/MMFRL이
# 사라질 수 있으므로 별도 폴더(additional_table/comparison)에 저장한다.
save_base_dir = '/data/rlawlsgurjh/work/gitlab/CUE/cue/results/table/benchmark_comparison'
os.makedirs(save_base_dir, exist_ok=True)

# results_table.ipynb 7개 + ADMET-Print, MMFRL
model_order = ['DeepDTA', 'GraphDTA', 'TEFDTA', 'HMM-DTA', 'ADMET-Print', 'ADMET-AI', 'MMFDL', 'MMFRL', 'CUE']

for d_name in datasets:
    df_subset = df_all[df_all['Dataset_Group'] == d_name]

    for task_type in ['regression']: # ['regression', 'classification']:
        # suffix = 'reg' if task_type == 'regression' else 'clf'
        metrics = regression_candidates # if task_type == 'regression' else classification_candidates
        targets = label_map[task_type]

        task_table_rows = []
        for target in targets:
            for model in model_order:
                m_data = df_subset[(df_subset['Target'] == target) & (df_subset['Modality'] == model)]
                row = {'Target': target, 'Model': model}

                for m in metrics:
                    # 지표 이름 표시 (AUROC, AUPRC 등)
                    m_label = 'AUROC' if m == 'auc_roc' else 'AUPRC' if m == 'auprc' else m.upper()
                    m_header = f"{m_label}( {metric_direction.get(m, '')} )"

                    if not m_data.empty and m in m_data.columns:
                        mean_val = m_data[m].values[0]
                        std_val = m_data[f"{m}_std"].values[0] if f"{m}_std" in m_data.columns else 0.0
                        # mean ± std (셋째 자리까지)
                        row[m_header] = f"{mean_val:.3f} ± {std_val:.3f}"
                    else:
                        row[m_header] = "N/A"

                if model == 'CUE' or any(row[f"{'AUROC' if m == 'auc_roc' else 'AUPRC' if m == 'auprc' else m.upper()}( {metric_direction.get(m, '')} )"] != "N/A" for m in metrics):
                    task_table_rows.append(row)

        g_name = 'DIS' if d_name == 'Kd' else 'INH'

        if task_table_rows:
            res_df = pd.DataFrame(task_table_rows)
            save_path = os.path.join(save_base_dir, f"{g_name}_results.csv")
            res_df.to_csv(save_path, index=False, encoding='utf-8-sig', sep='\t')
            print(f"📄 통합 결과 저장 완료: {save_path}")

display(res_df)

📄 통합 결과 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/table/benchmark_comparison/DIS_results.csv
📄 통합 결과 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/table/benchmark_comparison/INH_results.csv


,Target,Model,RMSE( ↓ ),R2( ↑ ),PCC( ↑ )
0,PI,DeepDTA,0.257 ± 0.025,-0.636 ± 0.304,0.411 ± 0.088
1,PI,GraphDTA,0.266 ± 0.030,-0.754 ± 0.351,0.371 ± 0.036
2,PI,TEFDTA,0.285 ± 0.017,-1.010 ± 0.181,0.324 ± 0.034
3,PI,HMM-DTA,0.192 ± 0.012,0.081 ± 0.150,0.575 ± 0.062
4,PI,ADMET-Print,0.187 ± 0.004,0.134 ± 0.052,0.471 ± 0.081
5,PI,ADMET-AI,0.169 ± 0.004,0.288 ± 0.082,0.556 ± 0.059
6,PI,MMFDL,0.188 ± 0.005,0.129 ± 0.067,0.386 ± 0.067
7,PI,MMFRL,0.170 ± 0.004,0.287 ± 0.062,0.588 ± 0.036
8,PI,CUE,0.163 ± 0.004,0.344 ± 0.053,0.597 ± 0.040
9,Ssel,DeepDTA,0.506 ± 0.053,-0.357 ± 0.330,0.609 ± 0.065
